# Lesson 03 — Measuring a drift of 0.1 arcsecond per orbit

**Phase 3 of the gravsim project: the measurement toolkit.**

Lesson 02 showed that the planets turn Mercury's ellipse by about $530''$ per century, with a
wobble on top. The relativistic effect we are after is $43''$ per century: one tenth of an
arcsecond per orbit, $5\times10^{-7}$ radians. This lesson builds the instrument that can see it.

By the end you should be able to

1. define the osculating elements and the eccentricity (Laplace–Runge–Lenz) vector, and say
   why the *longitude of perihelion* $\varpi$ is the quantity to track;
2. measure a precession rate three ways (dense series, perihelion passages, run-vs-run
   difference) and quote an uncertainty;
3. validate the instrument on a signal of known size — an oblate Sun — before pointing it at
   an unknown one;
4. explain why the N-body rate measured over ten years is not the textbook century value.

## 1. The physics: osculating elements and the eccentricity vector

**Osculating elements.** At any instant the position and velocity of a planet relative to the
Sun define a unique Kepler ellipse — the one it *would* follow if all other forces switched off
now. Its elements $(a, e, i, \Omega, \omega, \nu)$ are the *osculating* elements. On a pure
two-body orbit they are constant; with perturbations they vary slowly.

**The eccentricity vector.** With $\mu = GM_\odot + GM_{\rm planet}$ and
$\mathbf h = \mathbf r\times\mathbf v$,

$$ \mathbf e \;=\; \frac{\mathbf v\times\mathbf h}{\mu} - \frac{\mathbf r}{r} $$

has magnitude $e$ and points from the Sun to the perihelion. It is the Laplace–Runge–Lenz
vector divided by $\mu$, an exact constant of the Kepler problem; its direction *is* the
perihelion, available at every sample, not just when the planet passes through it.

**Why $\varpi$ and not $\omega$.** The argument of perihelion $\omega$ is measured from the
ascending node, and for an orbit of small inclination the node is badly defined: $\Omega$ and
$\omega$ individually jump around while their sum does not. The *longitude of perihelion*

$$ \varpi = \Omega + \omega $$

(a "dog-leg" angle, half in the ecliptic, half in the orbit plane) is continuous and
well-conditioned as $i\to0$. gravsim computes it from $\mathbf e$ and the node vector
(`gravsim.analysis.elements.perihelion_longitude`).

**Secular versus periodic.** A perturbed $\varpi(t)$ has a *secular* part, $\dot\varpi\,t$,
and *periodic* parts with the synodic periods of the perturbers. We want $\dot\varpi$. Three
estimators:

1. **dense fit** — ordinary least squares on the unwrapped $\varpi(t)$ at every sample;
2. **passages** — the same fit on $\varpi$ sampled only at perihelion passages (local minima of
   $r$, refined by a parabola through three samples);
3. **differencing** — subtract the $\varpi(t)$ of a reference run on the *same time grid* with
   the *same integrator* and fit the difference. Periodic terms common to both runs cancel, and
   so does the integrator's own spurious precession.

The OLS slope error $\sigma_{\dot\varpi} = \sqrt{\sum \text{res}^2/(n-2)/\sum(t-\bar t)^2}$
assumes white residuals; with periodic residuals it is a *nominal* figure, to be read with care.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from gravsim import constants as C
from gravsim.analysis import (
    fit_secular_rate,
    lrl_vector,
    perihelion_longitude,
    perihelion_longitude_series,
    perihelion_passages,
    perihelion_rate_difference,
    propagate,
    state_to_elements,
    wrap_angle,
)
from gravsim.data.horizons import load_j2000
from gravsim.forces import accel_newton, combine
from gravsim.integrators import integrate, integrate_scipy
from gravsim.viz import figure_grid, plot_precession

plt.rcParams["figure.dpi"] = 100


def daily_grid(t0, years):
    return t0 + np.arange(int(round(years * C.JULIAN_YEAR_DAYS)) + 1, dtype=float)


def rates(traj, body="Mercury"):
    """(dense, dense_err, passages) secular rates of varpi in arcsec per Julian century."""
    t, w = perihelion_longitude_series(traj, body)
    fit = fit_secular_rate(t, w)
    tp, wp = perihelion_passages(traj, body)
    return fit.rate_arcsec_per_century, fit.rate_err_arcsec_per_century, fit_secular_rate(tp, wp).rate_arcsec_per_century

## 2. Sanity check: a Kepler orbit does not precess

Propagate Mercury analytically (`propagate` solves Kepler's equation) for ten years and
compute $\mathbf e$ and $\varpi$ at every day: the direction must be constant to roundoff.
Then do the same with the DOP853 run of Lesson 01: the numerical noise floor.

In [ ]:
two = load_j2000().subset(("Sun", "Mercury")).to_barycentric()
r0, v0 = two.relative_to("Sun")
i_m = two.index("Mercury")
mu = float(two.gm.sum())
t_grid = daily_grid(two.t, 10.0)
r_k, v_k = propagate(r0[i_m], v0[i_m], mu, t_grid - two.t)  # (T, 3) analytic
e_vec = lrl_vector(r_k, v_k, mu)
varpi_k = perihelion_longitude(r_k, v_k, mu)
print(f"analytic Kepler: |e| = {np.linalg.norm(e_vec, axis=1).mean():.6f} ± {np.linalg.norm(e_vec, axis=1).std():.1e}")
print(f"                 ϖ spread = {np.degrees(varpi_k.max() - varpi_k.min()) * 3600:.2e} arcsec over 10 yr")

newton2 = integrate_scipy(two, accel_newton, t_grid)
dense, err, passages = rates(newton2)
print(f"DOP853 2-body:   dense fit {dense:+.4f} ± {err:.4f} arcsec/cy, passages {passages:+.4f} arcsec/cy")

## 3. The instrument on the real wobble: Mercury in the nine-body run

Now the full solar system (Newton only, DOP853, ten years). The dense series shows the
periodic terms clearly; the perihelion passages (one point every 88 days) sit on the same
trend. Both fits give the same rate within the nominal error.

In [ ]:
state = load_j2000()
t_grid = daily_grid(state.t, 10.0)
newton = integrate_scipy(state, accel_newton, t_grid)
t, w = perihelion_longitude_series(newton, "Mercury")
fit = fit_secular_rate(t, w)
tp, wp = perihelion_passages(newton, "Mercury")
fit_p = fit_secular_rate(tp, wp)

fig, axes = figure_grid(2, panel=(5.6, 3.8))
plot_precession(t, w, fit.rate, ax=axes[0], label="dense series (daily)")
s = (np.unwrap(wp) - np.unwrap(w)[0]) * C.ARCSEC_PER_RAD
axes[0].plot((tp - t[0]) / C.JULIAN_YEAR_DAYS, s, "o", ms=3, color="k", label="perihelion passages")
axes[0].legend(fontsize=8)
axes[0].set_title("Mercury, Sun + 8 planets, Newton: ϖ(t)")
axes[1].plot((t - t[0]) / C.JULIAN_YEAR_DAYS, fit.residuals * C.ARCSEC_PER_RAD, lw=0.7)
axes[1].set_xlabel("t [yr]"); axes[1].set_ylabel("residual [arcsec]"); axes[1].grid(True, lw=0.3)
axes[1].set_title("residuals of the linear fit: the periodic terms")
print(f"dense:    {fit.rate_arcsec_per_century:7.1f} ± {fit.rate_err_arcsec_per_century:.1f} arcsec/cy")
print(f"passages: {fit_p.rate_arcsec_per_century:7.1f} ± {fit_p.rate_err_arcsec_per_century:.1f} arcsec/cy  ({tp.size} passages)")
print(f"first passages at t = {tp[:3].round(2)} d  (spacing {np.diff(tp).mean():.3f} d = anomalistic period)")

## 4. Calibrating on a signal of known size: an oblate Sun

Before pointing the instrument at GR we test it on a perturbation whose answer we can derive
by hand. A central body flattened by its rotation has a quadrupole moment $J_2$, and the
potential becomes

$$ U = -\frac{GM}{r}\left[1 - J_2\left(\frac{R}{r}\right)^2 P_2(\cos\theta)\right],
\qquad P_2(x) = \tfrac12(3x^2-1), $$

with $\theta$ the colatitude from the symmetry axis. Taking $-\nabla U$ (axis along $z$):

$$
\mathbf a_{J_2} = \frac{3}{2}\,J_2\,\frac{GM R^2}{r^5}
\begin{pmatrix} x\,(5z^2/r^2 - 1)\\ y\,(5z^2/r^2 - 1)\\ z\,(5z^2/r^2 - 3)\end{pmatrix}.
$$

Averaged over an orbit this produces the classical secular rates (e.g. Vallado, *Fundamentals of
Astrodynamics*, §9.6), with $n = 2\pi/P$ and $p = a(1-e^2)$:

$$
\dot\Omega = -\tfrac32\,n J_2 \Big(\tfrac{R}{p}\Big)^2 \cos i, \qquad
\dot\omega = \tfrac34\,n J_2 \Big(\tfrac{R}{p}\Big)^2 (5\cos^2 i - 1),
$$

so $\dot\varpi = \dot\Omega + \dot\omega = \tfrac34\,n J_2 (R/p)^2\,(5\cos^2 i - 1 - 2\cos i)$,
which is $\tfrac32\,n J_2 (R/p)^2$ for an equatorial orbit.

The real Sun has $J_2 \approx 2.2\times10^{-7}$ (helioseismology): for Mercury that is only
$0.03''$/century. For the calibration we exaggerate it to $J_2 = 10^{-3}$, which should give
$\approx120''$/century — a signal of the same order as the GR term, *and* of known size. (For
simplicity the symmetry axis is the ecliptic pole; the real solar axis is tilted 7°.)

Note how a force law is written: any function `accel(pos, vel, gm, t) -> (N, 3)` is an
`AccelFn`, and `combine` adds force laws together.

In [ ]:
def make_j2(j2, radius, central=0):
    """Quadrupole acceleration of an oblate central body (symmetry axis = z)."""
    def accel_j2(pos, vel, gm, t=0.0):
        r = pos - pos[central]
        r2 = np.einsum("ij,ij->i", r, r)
        r2[central] = 1.0
        k = 1.5 * j2 * gm[central] * radius**2 / (r2 * r2 * np.sqrt(r2))  # 3/2 J2 GM R^2 / r^5
        z2 = r[:, 2] ** 2 / r2
        acc = np.empty_like(pos)
        acc[:, 0] = k * r[:, 0] * (5.0 * z2 - 1.0)
        acc[:, 1] = k * r[:, 1] * (5.0 * z2 - 1.0)
        acc[:, 2] = k * r[:, 2] * (5.0 * z2 - 3.0)
        acc[central] = 0.0
        return acc
    return accel_j2


def j2_rate_arcsec_per_century(j2, radius, a, e, inc, gm):
    n = math.sqrt(gm / a**3)
    p = a * (1.0 - e * e)
    ci = math.cos(inc)
    rate = 0.75 * n * j2 * (radius / p) ** 2 * (5.0 * ci * ci - 1.0 - 2.0 * ci)  # rad/day
    return rate * C.ARCSEC_PER_RAD * C.JULIAN_CENTURY_DAYS


el = state_to_elements(r0[i_m], v0[i_m], mu)
J2_BIG = 1e-3
predicted = j2_rate_arcsec_per_century(J2_BIG, C.R_SUN_AU, float(el["a"]), float(el["e"]), float(el["inc"]), mu)
oblate = combine(accel_newton, make_j2(J2_BIG, C.R_SUN_AU, central=two.index("Sun")))
t_grid2 = daily_grid(two.t, 10.0)
j2run = integrate_scipy(two, oblate, t_grid2)

dense, err, passages = rates(j2run)
diff, diff_err = perihelion_rate_difference(j2run, newton2, "Mercury")
print(f"analytic secular rate:        {predicted:8.2f} arcsec/cy")
print(f"dense fit:                    {dense:8.2f} ± {err:.2f}")
print(f"perihelion passages:          {passages:8.2f}")
print(f"differenced against Newton:   {diff:8.2f} ± {diff_err:.2f}")

In [ ]:
t2, w2 = perihelion_longitude_series(j2run, "Mercury")
_, wn = perihelion_longitude_series(newton2, "Mercury")
fig, axes = figure_grid(2, panel=(5.6, 3.8))
plot_precession(t2, w2, fit_secular_rate(t2, w2).rate, ax=axes[0], label=f"Newton + J2 = {J2_BIG:g}")
axes[0].set_title("Sun–Mercury with an oblate Sun: ϖ(t)")
d = wrap_angle(w2 - wn)
plot_precession(t2, d, fit_secular_rate(t2, d).rate, ax=axes[1], label="(Newton + J2) minus Newton")
axes[1].set_title("(Newton + J2) minus Newton: the J2 term alone")

All three estimators agree with the analytic secular rate to better than a tenth of a percent.
The instrument is calibrated. Note that the difference panel still oscillates: the wobble
you see there is the short-period part of the $J_2$ perturbation *itself*, present in only
one of the two runs. Differencing removes what the runs have in **common** — here just the
integrator's drift; in §5 it will be the planetary wobble.

Now the *real* $J_2$. Its $0.027''$/century is of the same order as the integrator's own
spurious rate ($+0.0012''$/cy, §2), and the dense fit returns the *sum* of the two. The
differenced series returns the analytic value to four decimals, because the reference run
cancels everything the two runs have in common — the integrator's drift included. (The
nominal OLS errors are far too small here: the residuals are periodic, not white.)

In [ ]:
J2_SUN = 2.2e-7
real = combine(accel_newton, make_j2(J2_SUN, C.R_SUN_AU, central=two.index("Sun")))
j2real = integrate_scipy(two, real, t_grid2)
predicted_real = j2_rate_arcsec_per_century(J2_SUN, C.R_SUN_AU, float(el["a"]), float(el["e"]), float(el["inc"]), mu)
dense, err, _ = rates(j2real)
diff, diff_err = perihelion_rate_difference(j2real, newton2, "Mercury")
print(f"real solar J2 = {J2_SUN:g}: analytic {predicted_real:.4f} arcsec/cy")
print(f"  dense fit    {dense:+.4f} ± {err:.5f}   (= signal + the integrator's +0.0012)")
print(f"  differenced  {diff:+.4f} ± {diff_err:.5f}   (the signal alone)")

## 5. Differencing in the N-body case

The same trick with the planets present. The dense $\varpi(t)$ of the nine-body run wobbles
by tens of arcseconds (Lesson 02); the difference between the runs with and without the
(exaggerated) $J_2$ is as clean as in the two-body case, and gives the same rate. This is the
design of every GR measurement in Lesson 04: **same grid, same integrator, one force term
switched on**.

In [ ]:
oblate9 = combine(accel_newton, make_j2(J2_BIG, C.R_SUN_AU, central=state.index("Sun")))
j2run9 = integrate_scipy(state, oblate9, t_grid)
diff9, err9 = perihelion_rate_difference(j2run9, newton, "Mercury")
dense9, denseerr9, _ = rates(j2run9)
print(f"9-body, dense fit with J2:            {dense9:7.1f} ± {denseerr9:.1f} arcsec/cy  (planets + J2)")
print(f"9-body, dense fit without J2 (sec 3): {fit.rate_arcsec_per_century:7.1f} ± {fit.rate_err_arcsec_per_century:.1f}")
print(f"difference of the two runs:           {diff9:7.2f} ± {err9:.2f}  (analytic J2 rate {predicted:.2f})")

## 6. The textbook number: 531.6 arcseconds per century

Our ten-year dense fit of the nine-body run gave about $570''$/cy, not $531.6$. Nothing is
wrong: an OLS slope over a span shorter than the long-period terms (the Jupiter–Saturn
"great inequality" has a period of 900 years; the Venus–Earth terms, 8 and 240 years) is
biased by whichever phase of those terms the span happens to cover. The *century* value is
what the textbooks quote. Run it (DOP853 nine-body for 100 years takes a few seconds).

In [ ]:
t_cy = daily_grid(state.t, 100.0)
newton_cy = integrate_scipy(state, accel_newton, t_cy)
t, w = perihelion_longitude_series(newton_cy, "Mercury")
fit_cy = fit_secular_rate(t, w)
tp, wp = perihelion_passages(newton_cy, "Mercury")
fit_cy_p = fit_secular_rate(tp, wp)
fig, axes = figure_grid(1, panel=(7.5, 4.0))
plot_precession(t, w, fit_cy.rate, ax=axes[0], label="Newton, Sun + 8 planets, dense")
axes[0].set_title("Mercury's longitude of perihelion over a century (planets only)")
print(f"dense fit, 100 yr:   {fit_cy.rate_arcsec_per_century:.1f} ± {fit_cy.rate_err_arcsec_per_century:.1f} arcsec/cy")
print(f"passages, 100 yr:    {fit_cy_p.rate_arcsec_per_century:.1f}")
print(f"Clemence (1947):     531.6   (Moon, asteroids, Pluto omitted here: ~0.1; OLS bias of long-period terms: ~2)")

## 7. Summary

* Track the **longitude of perihelion** $\varpi = \Omega+\omega$ from the eccentricity
  vector at every sample.
* Three estimators: dense fit, perihelion passages, and the **run-vs-run difference**, which
  cancels periodic terms and integrator error alike.
* Calibrated on an oblate Sun: analytic and measured rates agree to a tenth of a percent;
  the real $J_2$ ($0.027''$/cy) is separated from the integrator's drift only by differencing.
* Absolute rates need a century; differences are clean after a decade. Every GR number in the
  next lesson is a difference.

## Exercises

1. Change `J2_BIG` to $10^{-4}$ and $10^{-2}$. Does the measured/analytic ratio stay at one?
   At which value does the first-order secular theory start to fail?
2. Tilt the symmetry axis of `make_j2` to the real solar axis (7.25° from the ecliptic pole,
   node at ecliptic longitude 75.8°). How does Mercury's $J_2$ rate change?
3. ★ Compute the rate from `perihelion_passages` with every second passage only. How does the
   nominal error scale with the number of passages?
4. ★ The dense-fit bias of §6: fit the residuals of the century run with a few sinusoids
   (periods 2.1, 5.9, 11.9 years ...) and identify which planets they belong to.